# FDR Benchmark: Learned Composite Score

Extends the NIST23 leave-one-out FDR benchmark to test whether a **learned multi-feature composite** reduces FDR beyond any single metric.

**Approach:** Percolator-style — train a classifier on (query, candidate) pairs to predict correct/wrong, using spectral features available in NIST23. Cross-validated on queries (not candidates) to prevent leakage.

**Features per (query, candidate) pair:**
- entropy_similarity, reverse_score, forward_score, ratio_profile
- delta_mda, query spectral entropy, n_matched peaks

**Per-query context:**
- n_candidates, candidate rank by entropy_sim

**Protocol:** Same as `fdr_benchmark_scores.ipynb` — 20k NIST23 [M+H]+, leave-one-out, 10 ppm, 0.05 Da MS2 tol.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ms_entropy as me
from bisect import bisect_left, bisect_right
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, roc_curve, auc
import warnings
warnings.filterwarnings('ignore')

MSP_PATH = '../data/reference_db/nist_protonated_sampled.msp'
PPM_TOL = 10
MS2_TOL = 0.05

## 1. Parse, preprocess, and run leave-one-out (same as fdr_benchmark_scores)

Reuses identical parsing and preprocessing. The key difference: we now store **per-candidate features**, not just per-metric bests.

In [ ]:
# ── Parse MSP (identical to fdr_benchmark_scores) ──
def parse_msp(path):
    spectra = []
    current = {}
    peaks = []
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                if current and peaks:
                    arr = np.array(peaks, dtype=np.float32)
                    base_peak = arr[:, 1].max()
                    arr = arr[arr[:, 1] >= 0.01 * base_peak]
                    if len(arr) > 0:
                        current['peaks'] = arr
                        spectra.append(current)
                current = {}
                peaks = []
                continue
            if ':' in line and not line[0].isdigit():
                key, val = line.split(':', 1)
                key = key.strip().lower()
                val = val.strip()
                if key == 'name': current['name'] = val
                elif key == 'precursormz': current['precursor_mz'] = float(val)
                elif key == 'inchikey': current['inchikey14'] = val[:14]
            elif line[0].isdigit():
                parts = line.split()
                if len(parts) >= 2:
                    peaks.append([float(parts[0]), float(parts[1])])
    if current and peaks:
        arr = np.array(peaks, dtype=np.float32)
        arr = arr[arr[:, 1] >= 0.01 * arr[:, 1].max()]
        if len(arr) > 0:
            current['peaks'] = arr
            spectra.append(current)
    return spectra

spectra = parse_msp(MSP_PATH)

# Compute entropy, sort, clean, weight
for s in spectra:
    s['entropy'] = float(me.calculate_spectral_entropy(s['peaks'], clean_spectrum=True, min_ms2_difference_in_da=MS2_TOL))

sorted_idx = np.argsort([s['precursor_mz'] for s in spectra])
spectra_sorted = [spectra[i] for i in sorted_idx]
mz_array = np.array([s['precursor_mz'] for s in spectra_sorted])

for s in spectra_sorted:
    mask = np.abs(s['peaks'][:, 0] - s['precursor_mz']) > MS2_TOL
    s['peaks_clean'] = s['peaks'][mask]
    p = s['peaks_clean']
    if len(p) > 0:
        cleaned = me.clean_spectrum(p, min_ms2_difference_in_da=MS2_TOL)
        if len(cleaned) > 0:
            weighted = np.array(me.apply_weight_to_intensity(cleaned), dtype=np.float64)
            s['w_mz'] = weighted[:, 0]
            s['w_int'] = weighted[:, 1]
            total = s['w_int'].sum()
            s['w_int_norm'] = s['w_int'] / total if total > 0 else s['w_int']
        else:
            s['w_mz'] = s['w_int'] = s['w_int_norm'] = np.empty(0)
    else:
        s['w_mz'] = s['w_int'] = s['w_int_norm'] = np.empty(0)

print(f'Parsed {len(spectra_sorted):,} spectra, {len(set(s["inchikey14"] for s in spectra_sorted)):,} compounds')

In [ ]:
# ── Leave-one-out: store per-candidate features ──
n = len(spectra_sorted)
candidate_rows = []  # list of dicts, one per (query, candidate) pair

for qi in range(n):
    query = spectra_sorted[qi]
    qmz = query['precursor_mz']
    qik = query['inchikey14']
    qpeaks = query['peaks_clean']

    if len(qpeaks) == 0 or len(query['w_mz']) == 0:
        continue

    mz_lo = qmz * (1 - PPM_TOL / 1e6)
    mz_hi = qmz * (1 + PPM_TOL / 1e6)
    lo_idx = bisect_left(mz_array, mz_lo)
    hi_idx = bisect_right(mz_array, mz_hi)

    n_cands = 0
    query_candidates = []

    for ci in range(lo_idx, hi_idx):
        if ci == qi:
            continue
        cand = spectra_sorted[ci]
        if len(cand['peaks_clean']) == 0 or len(cand['w_mz']) == 0:
            continue
        n_cands += 1

        # Entropy similarity
        esim = me.calculate_entropy_similarity(
            qpeaks, cand['peaks_clean'], ms2_tolerance_in_da=MS2_TOL, clean_spectra=True
        )

        # Peak matching
        matched_l = 0.0
        matched_q = 0.0
        q_raw_m, l_raw_m = [], []
        used = np.zeros(len(query['w_mz']), dtype=bool)
        for j in range(len(cand['w_mz'])):
            diffs = np.abs(query['w_mz'] - cand['w_mz'][j])
            candidates_idx = np.where((diffs <= MS2_TOL) & ~used)[0]
            if len(candidates_idx) > 0:
                best = candidates_idx[np.argmin(diffs[candidates_idx])]
                matched_l += cand['w_int_norm'][j]
                matched_q += query['w_int_norm'][best]
                q_raw_m.append(query['w_int'][best])
                l_raw_m.append(cand['w_int'][j])
                used[best] = True

        n_matched = len(q_raw_m)
        if n_matched >= 2:
            qa = np.array(q_raw_m)
            la = np.array(l_raw_m)
            ratio = float(np.dot(qa/np.linalg.norm(qa), la/np.linalg.norm(la)))
            log_ratios = np.log2((qa + 1e-8) / (la + 1e-8))
            max_dev = float(np.max(np.abs(log_ratios)))
        else:
            ratio = np.nan
            max_dev = np.nan

        query_candidates.append({
            'query_idx': qi,
            'entropy_sim': esim,
            'forward': matched_l,
            'reverse': matched_q,
            'ratio_profile': ratio,
            'max_deviation': max_dev,
            'n_matched': n_matched,
            'delta_mda': abs(qmz - cand['precursor_mz']) * 1000,
            'query_entropy': query['entropy'],
            'cand_entropy': cand['entropy'],
            'correct': int(cand['inchikey14'] == qik),
        })

    # Add n_candidates and rank to each candidate
    if query_candidates:
        sims = [c['entropy_sim'] for c in query_candidates]
        ranks = len(sims) - np.argsort(np.argsort(sims))  # rank 1 = highest
        for j, c in enumerate(query_candidates):
            c['n_candidates'] = n_cands
            c['esim_rank'] = int(ranks[j])
        candidate_rows.extend(query_candidates)

    if (qi + 1) % 2000 == 0:
        print(f'  {qi+1:,}/{n:,} done... ({len(candidate_rows):,} pairs so far)')

pairs = pd.DataFrame(candidate_rows)
print(f'\nFinished: {len(pairs):,} (query, candidate) pairs')
print(f'  Queries: {pairs["query_idx"].nunique():,}')
print(f'  Correct pairs: {pairs["correct"].sum():,} ({pairs["correct"].mean():.1%})')
print(f'  Median candidates/query: {pairs.groupby("query_idx").size().median():.0f}')

## 2. Train composite model via cross-validation

5-fold GroupKFold on query_idx — each query's candidates are entirely in train or test, never split. This prevents leakage from the same query appearing in both sets.

Two models:
- **Logistic regression** (simple, interpretable baseline)
- **GBM** (matches the annotation confidence model architecture)

In [ ]:
FEATURE_COLS = [
    'entropy_sim', 'forward', 'reverse', 'ratio_profile',
    'max_deviation', 'n_matched', 'delta_mda',
    'query_entropy', 'cand_entropy', 'n_candidates', 'esim_rank',
]

X = pairs[FEATURE_COLS].fillna(0).values
y = pairs['correct'].values
groups = pairs['query_idx'].values

# ── Cross-validated predictions ──
gkf = GroupKFold(n_splits=5)

# Logistic regression
lr_probs = np.zeros(len(pairs))
for train_idx, test_idx in gkf.split(X, y, groups):
    lr = LogisticRegression(max_iter=1000, C=1.0)
    lr.fit(X[train_idx], y[train_idx])
    lr_probs[test_idx] = lr.predict_proba(X[test_idx])[:, 1]

# GBM
gbm_probs = np.zeros(len(pairs))
for train_idx, test_idx in gkf.split(X, y, groups):
    gbm = GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
    gbm.fit(X[train_idx], y[train_idx])
    gbm_probs[test_idx] = gbm.predict_proba(X[test_idx])[:, 1]

pairs['lr_score'] = lr_probs
pairs['gbm_score'] = gbm_probs

# Pair-level AUC
lr_auc = roc_auc_score(y, lr_probs)
gbm_auc = roc_auc_score(y, gbm_probs)
esim_auc = roc_auc_score(y, pairs['entropy_sim'])

print(f'Pair-level AUC (correct vs wrong candidate):')
print(f'  Entropy similarity:  {esim_auc:.3f}')
print(f'  Logistic regression: {lr_auc:.3f}')
print(f'  GBM:                 {gbm_auc:.3f}')

# Feature importance (fit GBM on all data for importance)
gbm_full = GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
gbm_full.fit(X, y)
print(f'\nGBM feature importance:')
for col, imp in sorted(zip(FEATURE_COLS, gbm_full.feature_importances_), key=lambda x: -x[1]):
    print(f'  {col:20s}  {imp:.3f}')

## 3. Compute FDR curves for composite scores

For each query, find best correct and best wrong score using each metric. Then compute FDR = FP / (FP + TP) at each threshold.

In [ ]:
# ── Per-query best correct/wrong for each metric ──
METRICS = {
    'entropy_sim': 'Entropy similarity',
    'lr_score': 'Logistic regression',
    'gbm_score': 'GBM composite',
}
COLORS = {
    'entropy_sim': 'tab:blue',
    'lr_score': 'tab:orange',
    'gbm_score': 'tab:red',
}

# Also store query entropy for stratification
query_entropies = {}
for qi in pairs['query_idx'].unique():
    query_entropies[qi] = spectra_sorted[qi]['entropy']

query_results = []
for qi, g in pairs.groupby('query_idx'):
    correct = g[g['correct'] == 1]
    wrong = g[g['correct'] == 0]
    row = {
        'query_idx': qi,
        'entropy': query_entropies[qi],
        'has_correct': len(correct) > 0,
    }
    for m in METRICS:
        row[f'best_correct_{m}'] = correct[m].max() if len(correct) > 0 else -1.0
        row[f'best_wrong_{m}'] = wrong[m].max() if len(wrong) > 0 else -1.0
    query_results.append(row)

qdf = pd.DataFrame(query_results)
print(f'Queries: {len(qdf):,}')
print(f'  With correct candidate: {qdf["has_correct"].sum():,} ({qdf["has_correct"].mean():.1%})')

# ── FDR curves ──
thresholds = np.linspace(0, 1, 201)

def compute_fdr_curve(df_sub, metric, thresholds):
    bc = df_sub[f'best_correct_{metric}'].values
    bw = df_sub[f'best_wrong_{metric}'].values
    fdrs = []
    for t in thresholds:
        tp = np.sum(bc >= t)
        fp = np.sum((bw >= t) & (bc < t))
        fdr = fp / (fp + tp) if (fp + tp) > 0 else 0.0
        fdrs.append(fdr)
    return np.array(fdrs)

# ── FDR comparison plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: FDR vs threshold
ax = axes[0]
for m, label in METRICS.items():
    fdr = compute_fdr_curve(qdf, m, thresholds)
    ax.plot(thresholds, fdr, color=COLORS[m], linewidth=2, label=label)
ax.axhline(y=0.05, color='gray', linestyle=':', alpha=0.5, label='5% FDR')
ax.axhline(y=0.10, color='gray', linestyle='--', alpha=0.5, label='10% FDR')
ax.set_xlabel('Score threshold', fontsize=12)
ax.set_ylabel('FDR', fontsize=12)
ax.set_title('FDR vs Threshold', fontsize=13)
ax.set_xlim(0, 1)
ax.set_ylim(0, 0.5)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Right: ROC
ax = axes[1]
for m, label in METRICS.items():
    bc = qdf[f'best_correct_{m}']
    bw = qdf[f'best_wrong_{m}']
    has_any = (bc >= 0) | (bw >= 0)
    sub = qdf[has_any].copy()
    sub['best_score'] = np.maximum(sub[f'best_correct_{m}'], sub[f'best_wrong_{m}'])
    sub['label'] = (sub[f'best_correct_{m}'] >= sub[f'best_wrong_{m}']).astype(int)
    valid = sub['best_score'] > -0.5
    sub = sub[valid]
    fpr, tpr, _ = roc_curve(sub['label'], sub['best_score'])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=COLORS[m], linewidth=2, label=f'{label} ({roc_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('FPR', fontsize=12)
ax.set_ylabel('TPR', fontsize=12)
ax.set_title('ROC — Query-level top-1', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/fdr_benchmark_composite.png', dpi=150, bbox_inches='tight')
plt.show()

# Print key thresholds
print(f'\n{"Metric":<25s} {"FDR@0.7":>8} {"FDR@0.8":>8} {"FDR@0.9":>8} {"FDR@0.95":>8}')
print('-' * 62)
for m, label in METRICS.items():
    fdr = compute_fdr_curve(qdf, m, thresholds)
    vals = [fdr[np.argmin(np.abs(thresholds - t))] for t in [0.7, 0.8, 0.9, 0.95]]
    print(f'{label:<25s} {vals[0]:>8.3f} {vals[1]:>8.3f} {vals[2]:>8.3f} {vals[3]:>8.3f}')

## 4. FDR stratified by spectral entropy

In [ ]:
entropy_bins = [(0, 1, 'S=0-1'), (1, 2, 'S=1-2'), (2, 3, 'S=2-3'), (3, 99, 'S>3')]
bin_colors = ['tab:red', 'tab:blue', 'tab:green', 'tab:purple']

fig, axes = plt.subplots(1, len(METRICS), figsize=(6 * len(METRICS), 5), sharey=True)

for ax, (m, label) in zip(axes, METRICS.items()):
    for (lo, hi, elabel), color in zip(entropy_bins, bin_colors):
        mask = (qdf['entropy'] >= lo) & (qdf['entropy'] < hi)
        subset = qdf[mask]
        if len(subset) < 10:
            continue
        fdr = compute_fdr_curve(subset, m, thresholds)
        ax.plot(thresholds, fdr, color=color, linewidth=2, label=f'{elabel} (n={len(subset):,})')
    ax.axhline(y=0.05, color='gray', linestyle=':', alpha=0.5)
    ax.axhline(y=0.10, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Score threshold', fontsize=11)
    ax.set_title(label, fontsize=12)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 0.5)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('FDR', fontsize=12)
plt.suptitle('FDR by Spectral Entropy Level', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../figures/fdr_benchmark_composite_by_entropy.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. TP/FP counts and save

In [ ]:
for m, label in METRICS.items():
    print(f'\n--- {label} ---')
    print(f'{"Threshold":>10} {"TP":>8} {"FP":>8} {"FDR":>8} {"TPR":>8}')
    print('-' * 45)
    bc = qdf[f'best_correct_{m}'].values
    bw = qdf[f'best_wrong_{m}'].values
    n_with_correct = qdf['has_correct'].sum()
    for t in [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]:
        tp = np.sum(bc >= t)
        fp = np.sum((bw >= t) & (bc < t))
        fdr = fp / (fp + tp) if (fp + tp) > 0 else 0.0
        tpr = tp / n_with_correct if n_with_correct > 0 else 0.0
        print(f'{t:>10.2f} {tp:>8} {fp:>8} {fdr:>8.4f} {tpr:>8.4f}')

# Save
import os
out_dir = '../results/fdr_benchmark_composite'
os.makedirs(out_dir, exist_ok=True)
pairs.to_csv(f'{out_dir}/candidate_pairs.csv', index=False)
qdf.to_csv(f'{out_dir}/query_results.csv', index=False)
print(f'\nSaved to {out_dir}/')
print(f'  candidate_pairs.csv: {len(pairs):,} rows')
print(f'  query_results.csv: {len(qdf):,} rows')